In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Cargar el dataset
df = pd.read_csv("noticias_limpias_balanceadas2.csv")

df["label"].value_counts()


In [ ]:
# 1. Eliminar duplicados exactos
df = df.drop_duplicates()
df["label"].value_counts()

In [ ]:
# 2. Eliminar filas donde falte título, contenido o label
df = df.dropna(subset=["titulo", "contenido", "label"])
df["label"].value_counts()

In [ ]:
# 3. Normalizar espacios en blanco sin modificar mayúsculas ni signos
df["titulo"] = df["titulo"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["contenido"] = df["contenido"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["autor"] = df["autor"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["label"].value_counts()

In [ ]:
# 4. Quitar caracteres de control invisibles (saltos raros, tabulaciones)
df["titulo"] = df["titulo"].str.replace(r"[\x00-\x1F\x7F]", "", regex=True)
df["contenido"] = df["contenido"].str.replace(r"[\x00-\x1F\x7F]", "", regex=True)
df["label"].value_counts()

In [ ]:
import pandas as pd
import re

def fix_date_safe(x):
    if pd.isna(x):
        return pd.NaT

    s = str(x).strip()

    # Reemplazar guiones raros por guion normal
    s = re.sub(r"[–—−]", "-", s)

    # Eliminar caracteres invisibles
    s = re.sub(r"[\x00-\x1F\x7F\xa0]", "", s)

    # Formato dd-mm-aaaa (aceptando 1 o 2 dígitos)
    match = re.match(r"^(\d{1,2})-(\d{1,2})-(\d{4})$", s)
    if match:
        d, m, y = match.groups()
        return f"{int(d):02d}-{int(m):02d}-{y}"

    # Si no coincide, no eliminar → convertir luego a NaT
    return s

# Aplicar sin eliminar filas
df["fecha"] = df["fecha"].apply(fix_date_safe)

# Convertir a datetime REAL
df["fecha"] = pd.to_datetime(df["fecha"], format="%d-%m-%Y", errors="coerce")




In [ ]:
# 5. Estandarizar fechas sin modificar su contenido textual
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"])  # eliminar fechas inválidas
df["label"].value_counts()

In [ ]:
# 6. Filtrar contenido demasiado corto (suele ser basura)
df = df[df["contenido"].str.len() > 20]
df["label"].value_counts()

In [ ]:
# 7. Resetear índice
df = df.reset_index(drop=True)
df["label"].value_counts()

In [ ]:
df["label"].value_counts()


In [ ]:
# Guardar dataset limpio
df.to_csv("fake_news_clean.csv", index=False)